# Habitat Extraction from MAES Level 2 Land Cover Raster

This notebook takes a categorical land cover raster (MAES Level 2 classes) as
input and derives binary habitat masks (1 = habitat present, 0 = elsewhere)
for individual land cover classes.

**Land cover class codes (MAES Level 2):**

| Code | Class |
|------|-------|
| 101 | Urban (Settlements and other artificial areas) |
| 102 | Cropland |
| 103 | Grassland |
| 104 | Forest and woodlands |
| 105 | Heathland and shrub |
| 106 | Sparsely vegetated land |
| 107 | Wetlands |
| 200 | Rivers and lakes |

**Outputs:**
- `habitat_urban.tif` — binary raster, 1 where class == 101, 0 elsewhere
- `habitat_cropland.tif` — binary raster, 1 where class == 102, 0 elsewhere

The extraction logic is written as a small reusable function so you can
easily generate masks for any of the other classes (grassland, forest,
wetlands, etc.) by changing a single parameter.


## 1. Setup

In [ ]:
# If needed, install dependencies (uncomment):
# %pip install rasterio numpy

import os
from pathlib import Path

import numpy as np
import rasterio
from rasterio.enums import Resampling


## 2. Parameters

Edit these paths/values for your data.

In [ ]:
# --- Input ---
INPUT_RASTER = "landcover.tif"          # path to the input categorical land cover raster

# --- Output ---
OUTPUT_DIR = "."                        # directory where output GeoTIFFs will be written

# --- Land cover class lookup (MAES Level 2) ---
LAND_COVER_CLASSES = {
    101: "Urban (Settlements and other artificial areas)",
    102: "Cropland",
    103: "Grassland",
    104: "Forest and woodlands",
    105: "Heathland and shrub",
    106: "Sparsely vegetated land",
    107: "Wetlands",
    200: "Rivers and lakes",
}

# NoData handling: pixels equal to the source raster's nodata value are kept
# as 0 in the output mask (i.e. treated as "not this habitat"). Set to True
# if you would rather they be written out as a separate nodata value in the
# output instead of 0.
PRESERVE_SOURCE_NODATA = False


## 3. Reusable extraction function

In [ ]:
def extract_binary_habitat(
    input_path,
    class_code,
    output_path,
    preserve_source_nodata=False,
):
    """
    Create a binary GeoTIFF mask from a categorical land cover raster.

    Parameters
    ----------
    input_path : str or Path
        Path to the input categorical land cover raster.
    class_code : int
        The land cover class value to extract (e.g. 101 for Urban).
    output_path : str or Path
        Path where the output binary GeoTIFF will be written.
    preserve_source_nodata : bool
        If True, pixels that were nodata in the source raster are written
        as nodata (255) in the output instead of being collapsed to 0.

    Returns
    -------
    Path
        The path to the written output raster.
    """
    with rasterio.open(input_path) as src:
        data = src.read(1)
        profile = src.profile.copy()
        src_nodata = src.nodata

        # Binary mask: 1 where the pixel equals the target class, else 0
        mask = (data == class_code).astype(np.uint8)

        if preserve_source_nodata and src_nodata is not None:
            nodata_out = 255
            mask[data == src_nodata] = nodata_out
        else:
            nodata_out = None

        profile.update(
            dtype=rasterio.uint8,
            count=1,
            nodata=nodata_out,
            compress="lzw",
        )

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(mask, 1)

    n_pixels = int(mask[mask == 1].size) if not preserve_source_nodata else int((mask == 1).sum())
    print(f"Wrote {output_path}  |  class {class_code} ({LAND_COVER_CLASSES.get(class_code, 'unknown')})  |  {n_pixels} matching pixels")
    return output_path


## 4. Extract the urban habitat mask (class 101)

In [ ]:
urban_output = os.path.join(OUTPUT_DIR, "habitat_urban.tif")

extract_binary_habitat(
    input_path=INPUT_RASTER,
    class_code=101,
    output_path=urban_output,
    preserve_source_nodata=PRESERVE_SOURCE_NODATA,
)


## 5. Extract the cropland habitat mask (class 102)

In [ ]:
cropland_output = os.path.join(OUTPUT_DIR, "habitat_cropland.tif")

extract_binary_habitat(
    input_path=INPUT_RASTER,
    class_code=102,
    output_path=cropland_output,
    preserve_source_nodata=PRESERVE_SOURCE_NODATA,
)


## 6. (Optional) Quick visual sanity check

Plots the two output masks side by side. Requires `matplotlib`.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

with rasterio.open(urban_output) as src:
    axes[0].imshow(src.read(1), cmap="Reds", vmin=0, vmax=1)
    axes[0].set_title("Urban habitat mask (101)")
    axes[0].axis("off")

with rasterio.open(cropland_output) as src:
    axes[1].imshow(src.read(1), cmap="YlGn", vmin=0, vmax=1)
    axes[1].set_title("Cropland habitat mask (102)")
    axes[1].axis("off")

plt.tight_layout()
plt.show()


## 7. (Optional) Generate masks for any other class

Since `extract_binary_habitat` is parameterized by `class_code`, you can
loop over `LAND_COVER_CLASSES` to generate a mask for every class in one
go — useful if you later need grassland, forest, wetlands, etc. for the
same InVEST pollination workflow.


In [ ]:
# Example: generate masks for ALL classes (uncomment to run)

# for code, name in LAND_COVER_CLASSES.items():
#     safe_name = name.split(" (")[0].lower().replace(" ", "_")
#     out_path = os.path.join(OUTPUT_DIR, f"habitat_{safe_name}.tif")
#     extract_binary_habitat(INPUT_RASTER, code, out_path, PRESERVE_SOURCE_NODATA)
